<h1> Causal disovery individual cells

<h2> imports

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from Preprocessing.imbalanced_performance.cleaned_preprocess import IndividualProcess, CommonProcesses
from project_methods import ProjectMethods
from pathlib import Path
from causal_discovery.algos.notears import NoTears
import networkx as nx
import hashlib
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.CALM import calm
from causallearn.utils.GraphUtils import GraphUtils
from causal_discovery.algos.notears import NoTears
import matplotlib.image as mpimg
from causaldag import unknown_target_igsp
import causaldag as cd
import random
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
import algorithms.dotears as dotears

<h2> dataset

In [6]:
groups = [[9, 11, 13, 15, 22], [10, 12], [14], [17, 18, 19, 20, 21]]
cols = ['TempData', 'VoltageData', 'CurrentData', 'CycleIndex']
cell_dataset = IndividualProcess.getLongCellDataset()
multicausaldataset = CommonProcesses.envcausaldiscoverydataset(cell_dataset, cols, groups)
print(multicausaldataset)

{'0':           TempData  VoltageData  CurrentData  CycleIndex
0        24.041702     4.197761          0.0           1
1        23.955681     4.197635          0.0           1
2        23.926302     4.197824          0.0           1
3        23.979719     4.197635          0.0           1
4        24.001940     4.197635          0.0           1
...            ...          ...          ...         ...
2711537  22.857420     3.319781          0.0           9
2711538  22.857420     3.319725          0.0           9
2711539  22.857420     3.319683          0.0           9
2711540  22.857420     3.319831          0.0           9
2711541  22.857420     3.319834          0.0           9

[2260406 rows x 4 columns], '1':           TempData  VoltageData  CurrentData  CycleIndex
3601     23.878704     4.028611    -4.850105           1
3602     23.912833     4.022998    -4.848660           1
3603     24.009201     4.018773    -4.850067           1
3604     23.950279     4.014042    -4.848680    

<h2> Causal Discovery

<h3> PC-algorithm

In [ ]:
obs_data = multicausaldataset["1"]
labels = obs_data.columns.tolist()
print(labels)
# Convert to numpy array
data = obs_data.to_numpy()
print(data.shape)
print(labels)
n_samples = data.shape[0]
n_features = data.shape[1]
pc_result = pc(data)


In [ ]:
pydot_graph_pc = GraphUtils.to_pydot(pc_result.G, labels=labels)
print(pydot_graph_pc)

<h3> NoTears-algorithm

<h4> Per Goup

In [ ]:

dags = ProjectMethods.NoTearsLargeDataset(multicausaldataset["0"].to_numpy(), multicausaldataset["0"].columns.tolist(), '../Src/structures/1cells_char_2/images/multienv/NoTears/ByGroup')
print(dags)

In [ ]:
dag_list, cols = ProjectMethods.load_all_dags("../Src/structures/1cells_char_2/images/multienv/NoTears/ByGroup")           # from saved files
freq_matrix = ProjectMethods.dag_frequency_matrix(dag_list)
print("Matrix shape:", freq_matrix.shape)
print("Non-zero elements:", np.count_nonzero(freq_matrix))
print(freq_matrix)
ProjectMethods.plot_dag_frequency_heatmap(freq_matrix, cols, save_path="../Src/structures/1cells_char_2/images/multienv/NoTears/ByGroup/edge_frequency_heatmap.png")

<h4> Full-dataset

In [ ]:
dags = ProjectMethods.NoTearsLargeDataset(X_full=data, cols=cell_dataset.columns.tolist(), save_point='../Src/structures/1cells_char_2/images/multienv/NoTears/fulldata', subset_size=20000000)
print(dags)

In [ ]:
img_path = "../Src/structures/1cells_char_2/images/multienv/NoTears/fulldata/dag.png"

# Load and show
img = mpimg.imread(img_path)
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')  # hide axes
plt.show()

<h3> Third alg

In [ ]:
calm_result = calm(data)


In [ ]:
pydot_graph_calm = GraphUtils.to_pydot(calm_result['G'], labels=labels[0])
pydot_graph_calm.write_png('../Src/structures/1cells_char_2/images/multienv/calm.png')

In [ ]:
# ===============================
# 0. Imports
# ===============================
import numpy as np
from causaldag import (
    gsp,
    partial_correlation_suffstat,
    partial_correlation_test,
    MemoizedCI_Tester
)

# ===============================
# 1. Load / prepare YOUR data
# ===============================
# IMPORTANT:
# - rows = samples
# - columns = variables
# - no missing values
# - approximately linear + Gaussian

# 👉 REPLACE THIS with your data
samples = data   # shape: (n_samples, n_variables)

# OPTIONAL: standardize (often helps)
samples = (samples - samples.mean(axis=0)) / samples.std(axis=0)

# ===============================
# 2. Define variables / nodes
# ===============================
n_samples, n_variables = samples.shape

# Nodes must be labeled 0..p-1
nodes = set(range(n_variables))

# OPTIONAL: keep names for interpretation
var_names = [
    'TempData', 'VoltageData', 'CurrentData', 'CycleIndex', 'Group'
]  # 👉 replace with your variable names

# ===============================
# 3. Build sufficient statistics
# ===============================
# Uses covariance / partial correlations
suffstat = partial_correlation_suffstat(samples)

# ===============================
# 4. Conditional independence tester
# ===============================
# alpha = significance level
# 👉 TUNE THIS if graph is too dense / sparse
ci_tester = MemoizedCI_Tester(
    partial_correlation_test,
    suffstat,
    alpha=1e-3      # try: 1e-2, 1e-3, 1e-4
)

# ===============================
# 5. Run GSP
# ===============================
est_dag = gsp(nodes, ci_tester)

# ===============================
# 6. Inspect results
# ===============================

# Directed edges (i -> j)
print("Estimated edges:")
for i, j in est_dag.arcs:
    print(f"{var_names[i]} -> {var_names[j]}")

# ===============================
# 7. (Recommended) Look at CPDAG
# ===============================
# GSP returns ONE DAG in the equivalence class
# CPDAG shows which directions are identifiable

cpdag = est_dag.cpdag()

In [ ]:
data = multicausaldataset
data_for_dotears = {
    "obs": data["0"].values,
    "2_0": data["1"].values,
    "2_1": data["2"].values,
    "2_2": data["3"].values
}
for k, v in data_for_dotears.items():
    print(k, v.shape)

intervention_targets = {
    "0": 2,
    "1": 2,
    "2": 2,

}

obs (2260406, 4)
2_0 (158004, 4)
2_1 (34542, 4)
2_2 (258590, 4)


In [11]:
lambda1 = [0.01, 0.05, 0.1, 0.2, 0.5]
scaled = [True, False]
w_threshold = [0.0, 0.02, 0.05]
obs_only = [True, False]

In [12]:
doTEARS = dotears.DOTEARS(data_for_dotears)

In [13]:
W_est = doTEARS.fit()

In [14]:
threshold = 0.02
dag = (np.abs(W_est) > threshold).astype(int)
print(dag)
w_threshold = 0.02

[[0 0 0 0]
 [1 0 0 1]
 [1 0 0 1]
 [1 0 0 0]]


In [15]:
base_dir = "../Src/structures/1cells_char_2/images/multienv/DoTears/fulldata/"
for scaled in [False, True]:
    for lambda1 in [0.05, 0.1, 0.2]:
        model = dotears.DOTEARS(
            data_for_dotears,
            lambda1=lambda1,
            scaled=scaled,
            w_threshold=0.02
        )
        W = model.fit()
        print(f"scaled={scaled}, lambda={lambda1}")
        dag = (np.abs(W) > w_threshold).astype(int)
        # --- directory for group
        save_point = f"{base_dir}/{scaled}/{lambda1}"
        os.makedirs(save_point, exist_ok=True)
        ProjectMethods.makeAndSaveGraph(multicausaldataset["1"].columns.tolist(), dag, save_point)

scaled=False, lambda=0.05
scaled=False, lambda=0.1
scaled=False, lambda=0.2
scaled=True, lambda=0.05
scaled=True, lambda=0.1
scaled=True, lambda=0.2


<h3> GIES

In [ ]:
import gies

interventions = [
    [],
    [2],
    [2],
    [2]
]

data = multicausaldataset
data_for_gies = [
    data["0"].values,
    data["1"].values,
    data["2"].values,
    data["3"].values
]

nd_array = gies.fit_bic(data_for_gies, interventions)

/Users/christofferslettebo/anaconda3/envs/Master/lib/python3.10/site-packages/gies/scores/gauss_int_l0_pen.py:131: RuntimeWarning: divide by zero encountered in scalar divide
  gamma = 1 / omegas[j]
/Users/christofferslettebo/anaconda3/envs/Master/lib/python3.10/site-packages/gies/scores/gauss_int_l0_pen.py:134: RuntimeWarning: invalid value encountered in multiply
  - gamma
/Users/christofferslettebo/anaconda3/envs/Master/lib/python3.10/site-packages/gies/scores/gauss_int_l0_pen.py:134: RuntimeWarning: invalid value encountered in matmul
  - gamma
/Users/christofferslettebo/anaconda3/envs/Master/lib/python3.10/site-packages/gies/scores/gauss_int_l0_pen.py:171: RuntimeWarning: divide by zero encountered in log
  likelihood = -0.5 * n * (1 + np.log(sigma))
/Users/christofferslettebo/anaconda3/envs/Master/lib/python3.10/site-packages/gies/main.py:545: RuntimeWarning: invalid value encountered in scalar subtract
  valid_operators.append((new_score - old_score, x, y, T))


KeyboardInterrupt: 